<a href="https://colab.research.google.com/github/GAOYUEtianc/RLexperiments/blob/main/GANdiscriminator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Add a path to store our trained model
model_name = 'discriminator.pt'
path = F"/content/gdrive/My Drive/{model_name}" 

In [ ]:

from keras.datasets import mnist
import matplotlib.pyplot as plt
import numpy as np
from numpy import expand_dims

from keras.models import Model
from keras.models import load_model

from keras.optimizers import Adam

from keras.layers import Input
from keras.layers import Conv2D
from keras.layers import LeakyReLU
from keras.layers import Dropout
from keras.layers import Flatten
from keras.layers import Dense

In [ ]:
(X_train, y_train), (X_test, y_test) =  mnist.load_data()
# Data Preparation
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

In [ ]:
# Cut the image in X_test vertically in half 
def CutHalf(x):
    height,width = x.shape
    return [x[: , :width//2] , x[:, width//2:] ]
def PasteTogether(x_1, x_2):
    height, width = x_1.shape
    x = np.zeros((height, width*2))
    x[:,width//2] = x_1
    x[:, width//2:] = x_2
    return x
# Sort X_train according to labels
index_label = np.argsort(y_train)
X_train_sorted = X_train[index_label]
y_train_sorted = y_train[index_label]

X_train_half = np.zeros((2*len(X_train_sorted),28,14))
for i in range(0,2*len(X_train),2):
    X_train_half[i], X_train_half[i+1] = CutHalf(X_train_sorted[i//2])

X_train_later = []
# Generate fake images by pasting unmatched figures
for i in range(10):
    # Inverse left and right for images of label i
    for j in range(0, 1200, 2):
        X_train_later.append(np.hstack((X_train_half[12000*i+j+1],X_train_half[12000*i+j])))
    # collapse s.t. left is of label i, right is of another label
    exclude_i = np.delete(np.arange(10), i)
    for label in exclude_i:
        for j in range(0,1200, 2):
            X_train_later.append(np.hstack((X_train_half[12000*i+j],X_train_half[12000*label+j+1])))
X_train_later  = np.array(X_train_later)

# Cut the test set into half pieces, and shuffle them
X_test_half = np.zeros((2*len(X_test),28,14))
for i in range(0,2*len(X_test),2):
    X_test_half[i], X_test_half[i+1] = CutHalf(X_test[i//2])
# Shuffle the half pieces
np.random.shuffle(X_test_half)

In [ ]:
for i in range(800, 809):
    plt.subplot(331+i-800) # plot of 3 rows and 3 columns
    plt.axis('off') # turn off axis
    plt.imshow(X_train_later[i], cmap='gray') # gray scale

In [ ]:
# Combine the generated fake images with the original training dataset
X_train_double = np.concatenate((X_train, X_train_later), axis=0)
X_train_double = expand_dims(X_train_double, axis=-1)
print(X_train_double.shape)

In [ ]:
# expand each image dim to (28,28,1)
X_train = expand_dims(X_train, axis = -1)
#X_test = expand_dims(X_test, axis = -1)
# The first half dataset is true and labeled 1 , the later half (fake data) is labeled 0
y_train_binary = np.zeros(2*len(X_train))
for i in range(60000):
    y_train_binary[i] = 1

In [ ]:
# The discriminator model takes input an image and judge whether its a true image or not
kernel_size = (3,3)
strides_size = (2,2)
input = Input(shape = (28,28,1))
# First Layer
Encoder = Conv2D(128, kernel_size, strides_size, padding = 'same')(input)
Encoder = LeakyReLU(alpha=0.2)(Encoder)
# Second Layer
Encoder = Conv2D(128, kernel_size, strides_size, padding = 'same')(Encoder)
Encoder = LeakyReLU(alpha=0.2)(Encoder)
# Third Layer
Encoder = Conv2D(128, kernel_size, strides_size, padding = 'same')(Encoder)
Encoder = LeakyReLU(alpha=0.2)(Encoder)
# Feature maps 
Encoder = Flatten()(Encoder)
# Dropout features
Encoder = Dropout(0.4)(Encoder)
# Output layer is of dim 1 
outputLayer = Dense(1, activation='sigmoid')(Encoder)
# The discriminator model : input (28X28X1), output layer is dim 1
discriminator = Model(input, outputLayer)
discriminator.compile(loss='binary_crossentropy', optimizer = Adam(learning_rate = 0.001, beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-07,
    amsgrad=False,
    name="Adam"))

discriminator.fit(X_train_double, y_train_binary, epochs=10, batch_size=1000, verbose=2, validation_split=0.1)
discriminator.save(path)

In [ ]:
discriminator = load_model(path)

In [ ]:
# Cut the test set into half pieces, and shuffle them
import numpy as np
X_test_half = np.zeros((2*len(X_test),28,14))
for i in range(0,2*len(X_test),2):
    X_test_half[i], X_test_half[i+1] = CutHalf(X_test[i//2])
    
np.random.shuffle(X_test_half)

# see what's the prediction of fake images
X_test_temp = []
combinations = [(0,1),(2,3),(4,11),(6,7),(8,9),(10,5),(12,13),(14,15),(16,17)]
for indexTuple in combinations:
    l, r = indexTuple
    X_test_temp.append(np.hstack((X_test_half[l], X_test_half[r])))
X_test_temp = np.array(X_test_temp)
y_hat = discriminator.predict(X_test_temp)
print(y_hat)

In [ ]:
for i in range(0, 9):
    plt.subplot(331+i) # plot of 3 rows and 3 columns
    plt.axis('off') # turn off axis
    plt.imshow(X_test_temp[i], cmap='gray') # gray scale

In [ ]:
# the threshold of 'correct image' is set to be 0.95
thresh = 0.98
X_test_half = X_test_half.tolist()
# X_matched is to store the correct images and Y_hat is to store their scores
X_matched = []
Y_hat = []

In [ ]:
import pickle as pkl
# Add a path to store our mathed 'correct' figure data
output_name = 'matched_images_900.pkl'
path_matchedfigure = F"/content/gdrive/My Drive/{output_name}" 
fileObject = open(path_matchedfigure, 'wb')
thresh = 0.98
X_matched = X_matched.tolist()
# The left pointer is the current index for 'left half image', right pointer is the current index for 'right half image'
left_pointer = 0
right_num = 0
# Can run many tasks in parallel (starting from different 'left pointers', e.g., left_pointer = 0,100,200,etc)
while X_test_half[left_pointer] != None and len(X_matched)<900: 
    left = X_test_half[left_pointer]
    for right_pointer in range(left_pointer+1, len(X_test_half)):
        right =  X_test_half[right_pointer]
        img_temp = np.hstack((left, right))
        #print(img_temp.shape)
        img_temp= expand_dims(img_temp, axis=-1)
        img_temp= expand_dims(img_temp, axis= 0)
        #print(img_temp.shape)
        score = discriminator.predict(img_temp)
        if np.take(score,0) > thresh:
            X_matched.append(img_temp)
            Y_hat.append(score)
            X_test_half.pop(left_pointer)
            X_test_half.pop(right_pointer)
            right_num += 1
            print("num of correct img", right_num)
            break
        # Reverse the left and right
        img_temp = np.hstack((right,left))
        img_temp= expand_dims(img_temp, axis=-1)
        img_temp= expand_dims(img_temp, axis= 0)
        score = discriminator.predict(img_temp)
        if np.take(score,0) > thresh:
            X_matched.append(img_temp)
            Y_hat.append(score)
            X_test_half.pop(left_pointer)
            X_test_half.pop(right_pointer)
            right_num += 1
            print("num of correct img", right_num)
            break
    left_pointer += 1 
# write the saved imaged into a file
X_matched = np.array(X_matched)
pkl.dump(X_matched, fileObject)
fileObject.close()

In [ ]:
import pickle as pkl
output_name = 'matched_images_800.pkl'
path_matchedfigure = F"/content/gdrive/My Drive/{output_name}" 
fileObject = open(path_matchedfigure, 'rb')
img = pkl.load(fileObject)
img_plot = []
fileObject.close()
for i in range(800):
    img_plot.append(np.reshape(img[i], (28,28)))
for i in range(220,229):
    plt.subplot(331+i-220) # plot of 3 rows and 3 columns
    plt.axis('off') # turn off axis
    plt.imshow(img_plot[i], cmap = 'gray')